# Full bank · label, pair, draw and split (label keeper only)

Runtime: **CPU** (the free one is enough). Run it only if you are the **label keeper**: the person who holds the test labels (role F in PLAN.md §2) and takes no part in scoring, in choosing pairs or in setting thresholds. Nobody else keeps a copy of anything in the private folder; people who help with the label audit or the error tags delete theirs when done.

Before you start:
- The Drive folder from notebook 10 is shared with you (add a shortcut to it in My Drive) and complete for the domains below.
- Colab secret `HF_TOKEN`, from an account with access to Llama-3.1-8B, Llama-3.2-3B, Gemma-2-9B and Mistral-7B. Their tokenizers are needed for the length filter.

Three folders:
- `SHARED`: the answers from notebook 10. Only read.
- `PRIVATE`: **your own Drive, never shared.** Labeled answers, execution logs, test labels and the error-tag sheet.
- `COMMIT`: what goes into the repository: the bank, the validation labels and the manifests.

A step that fails stops the notebook. Only counts and hashes are printed, never a test label or a result for one answer. Do not open the private files with anyone who scores, selects or sets thresholds.

In [ ]:
I_AM_THE_LABEL_KEEPER = False   # set to True only if the text above describes you
DOMAINS = ["math", "code"]      # or just one of them
VERSION = "v1"                  # a corrected bank, for example after a failed label audit, is a new version: raise this
SHARED = "/content/drive/MyDrive/cs544_full_bank"
PRIVATE = f"/content/drive/MyDrive/cs544_full_bank_private/{VERSION}"
COMMIT = f"/content/drive/MyDrive/cs544_full_bank_to_commit/{VERSION}"
WORK = f"/content/work_{VERSION}"
BOX = f"/content/sandbox_{VERSION}"
DISCONNECT = True
BRANCH = "main"
assert I_AM_THE_LABEL_KEEPER, "only the label keeper runs this notebook (PLAN.md §2)"
FOLDERS = (SHARED, PRIVATE, COMMIT)
assert sum((a + "/").startswith(b + "/") for a in FOLDERS for b in FOLDERS) == 3, "the three folders must be separate"

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
%cd /content
!rm -rf /content/project && git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q math-verify==0.9.0 evalplus==0.3.1 transformers==5.19.0 sentencepiece tiktoken protobuf
!mkdir -p {PRIVATE} {COMMIT} {WORK} && chmod 700 {WORK}

import datetime, subprocess
from importlib.metadata import version
assert (version("math-verify"), version("evalplus")) == ("0.9.0", "0.3.1")  # the pilot's checkers
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
now = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
with open(f"{PRIVATE}/runs.txt", "a") as f:
    f.write(f"{now} notebook 11, commit {commit}, version {VERSION}, domains {DOMAINS}\n")
STOPPED = "stopped: see the message above"

## 1. Collect the answers
Stops if any generator is incomplete, or used another model commit, setting or question file.

In [ ]:
for d in DOMAINS:
    !python -m bank.generate_full merge --domain {d} --gen {SHARED}/gen --out {WORK}/{d}/generations.jsonl
    assert _exit_code == 0, STOPPED

## 2. Math labels
The boxed answer and Math-Verify must agree.

In [ ]:
if "math" in DOMAINS:
    !python -m bank.label --questions bank/full/questions_math.jsonl --generations {WORK}/math/generations.jsonl --out {WORK}/math/labeled.jsonl
    assert _exit_code == 0, STOPPED

## 3. Code labels
EvalPlus's base and plus tests, each program twice. These are model-written programs, so they run as a user who cannot read Drive, the labels or a Hugging Face login, with an empty environment; if that user could, the cell stops before running anything. Expect several hours on a free runtime with 2 CPUs; finished checks are copied to the private folder every 5 minutes, so after a disconnect run the notebook again and it continues.

In [ ]:
if "code" in DOMAINS:
    import os, shutil, subprocess, threading
    CHECKS, SAVED, LOGIN = f"{BOX}/evalplus_checks.jsonl", f"{PRIVATE}/code/evalplus_checks.jsonl", "/root/.cache/huggingface"
    os.makedirs(os.path.dirname(SAVED), exist_ok=True)

    def plain(path):  # root never follows a link that the sandbox user could have made
        return os.path.isfile(path) and not os.path.islink(path)

    def save_checks():
        try:
            if plain(CHECKS) and (not os.path.isfile(SAVED) or os.path.getsize(CHECKS) > os.path.getsize(SAVED)):
                shutil.copyfile(CHECKS, SAVED)
        except OSError as exc:  # Drive hiccup: try again next time
            print("could not save the finished checks:", exc)

    save_checks()  # what an interrupted run in this runtime left
    if os.path.isdir(BOX):
        shutil.rmtree(BOX)
    os.makedirs(BOX)
    shutil.copyfile(f"{WORK}/code/generations.jsonl", f"{BOX}/generations.jsonl")
    if os.path.isfile(SAVED):
        shutil.copyfile(SAVED, CHECKS)
    !chown -R nobody {BOX}
    assert _exit_code == 0, STOPPED
    if os.path.isdir(LOGIN):
        os.chmod(LOGIN, 0o700)
    for folder in (PRIVATE, SHARED, COMMIT, WORK, LOGIN):
        assert subprocess.run(["runuser", "-u", "nobody", "--", "ls", folder], capture_output=True).returncode != 0, f"the sandbox user can read {folder}: stop"
    assert subprocess.run(["runuser", "-u", "nobody", "--", "touch", f"{BOX}/labeled.jsonl"]).returncode == 0, "the sandbox user cannot write its own folder"

    finished = threading.Event()
    def keep_saving():
        while not finished.wait(300):
            save_checks()
    saver = threading.Thread(target=keep_saving, daemon=True)
    saver.start()
    !runuser -u nobody -- env -i HOME={BOX} LANG=C.UTF-8 PATH=/usr/local/bin:/usr/bin:/bin python -m bank.label_evalplus --questions bank/full/questions_code.jsonl --generations {BOX}/generations.jsonl --out {BOX}/labeled.jsonl --progress {CHECKS}
    code_exit = _exit_code
    finished.set()
    saver.join()
    save_checks()
    assert code_exit == 0, STOPPED
    for name in ("labeled.jsonl", "labeled_manifest.json"):
        assert plain(f"{BOX}/{name}"), f"{name} is missing or is a link"
        shutil.copyfile(f"{BOX}/{name}", f"{WORK}/code/{name}")

## 4. Pairs, length filter, draw and split

In [ ]:
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
for d in DOMAINS:
    !python -m bank.make_pairs --split full --questions bank/full/questions_{d}.jsonl --generations {WORK}/{d}/generations.jsonl --labeled {WORK}/{d}/labeled.jsonl --out {WORK}/{d}
    assert _exit_code == 0, STOPPED
    !python -m bank.token_lengths {WORK}/{d}/full_bank.jsonl --out {WORK}/{d}/too_long.json
    assert _exit_code == 0, STOPPED
    !python -m bank.draw_full --domain {d} --work {WORK}/{d} --questions bank/full/questions_{d}.jsonl --generations {WORK}/{d}/generations.jsonl
    assert _exit_code == 0, STOPPED
    !python -m bank.split_full split --domain {d} --work {WORK}/{d} --out {COMMIT}/bank/full/{d} --private {PRIVATE}/{d}
    assert _exit_code == 0, STOPPED
    !mkdir -p {PRIVATE}/{d}/work && cp -r {WORK}/{d}/. {PRIVATE}/{d}/work/
    assert _exit_code == 0, STOPPED

## 5. Math error tags
Writes a sheet of 200 wrong math test answers to the private folder (PLAN.md §2). When `error_type` is filled in, set `APPLY_TAGS` to True and run the first two code cells, the second cell of this section and section 6. Do this before the bank freeze. If the sheet already exists, it is left alone.

In [ ]:
if "math" in DOMAINS:
    !python -m bank.split_full error-sheet --out {COMMIT}/bank/full/math --private {PRIVATE}/math

In [ ]:
APPLY_TAGS = False
if APPLY_TAGS:
    !python -m bank.split_full apply-tags --out {COMMIT}/bank/full/math --private {PRIVATE}/math
    assert _exit_code == 0, STOPPED

## 6. What to commit
The bank freeze commits the files listed here, after the label audit has passed. The test labels stay in the private folder until they are joined, once, after the preregistration commit.

In [ ]:
import hashlib, shutil
from pathlib import Path
public = Path(COMMIT) / "bank/full"
for path in [p for d in DOMAINS for p in sorted(Path(SHARED).glob(f"gen/*_{d}/generations_manifest.json"))] + [Path(SHARED) / "api_spend.json"]:
    target = public / path.relative_to(SHARED)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(path, target)
files = sorted(p for p in Path(COMMIT).rglob("*") if p.is_file())
private_files = [p.name for p in files if p.name not in ("bank.jsonl", "labels_validation.jsonl", "manifest.json", "generations_manifest.json", "api_spend.json")]
assert not private_files, f"must not be committed: {private_files}"
for p in files:
    print(hashlib.sha256(p.read_bytes()).hexdigest(), p.relative_to(COMMIT))
for d in DOMAINS:
    print(hashlib.sha256((Path(PRIVATE) / d / "labels_test.jsonl").read_bytes()).hexdigest(), d, "labels_test.jsonl (private, hash only)")

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
if DISCONNECT:
    runtime.unassign()